# clikernel

> Connect LLMs to persistent gateway-hosted Jupyter kernels as concise text, over MCP or a plain stream protocol


`clikernel` gives LLM agents persistent kernel sessions over MCP or a plain stream protocol. Imports, variables, and results remain available between calls. Agents can create kernels or attach to existing ones, including a user's live solveit kernel.

[rustygate](https://github.com/AnswerDotAI/rustygate) hosts the kernels. Clikernel routes requests to local or named remote gateways and supplies startup code to Python kernels.

New kernels close with their creating conversation by default. Attaching to an existing kernel does not claim ownership. Use `autoclose=false` on a persistent gateway to keep a new kernel for later conversations. See [core](https://AnswerDotAI.github.io/clikernel/core.html) for configuration and lifecycle details.


## Install

```sh
pip install clikernel
```

This installs rustygate and ipymini. No service setup is required for a conversation-local kernel: clikernel starts a private gateway if it cannot find one. To retain kernels across conversations, run a resident gateway, for example through launchd or systemd:

```sh
rustygate --port 8787
```


## Use with an MCP host

Register the stdio server with your MCP host. For Claude Code:

```sh
claude mcp add clikernel -- clikernel-mcp
```

Choose a kernel, then execute code:

```python
create(kernel="py")
exec(code="6*7")
```

Use `kernel="luau"` for bundled Luau, or `kernel="apl"` for APL when `basedpl` is installed in the gateway's environment. `create` accepts any kernelspec the gateway finds. `exec` never creates or switches kernels. See [MCP](https://AnswerDotAI.github.io/clikernel/mcp.html) for named gateways, dialog bindings, and other tools.


## Configuration

Three optional files in `$XDG_CONFIG_HOME/clikernel/` configure the router, usually under `~/.config/clikernel/`, or under the directory `clikernel-mcp --cfgdir` names:

- `startup.py` runs in each Python kernel clikernel creates, with `__file__` set to its path. Its output appears in the reply announcing the kernel unless `--quiet` is set.
- `inspectors.py` installs Python cell inspectors after startup. Define `inspect`, a list named `inspectors`, or both. Each inspector runs once before a cell: a one-argument inspector takes the cell's AST, and a two-argument inspector takes the AST and raw source. Return a string to print a note before the output. Raise the provided `RuleBlock` to block execution. Other exceptions produce a warning and allow the cell to run. See [examples/inspectors.py](examples/inspectors.py).
- `gateways.toml` names remote gateways and configures authentication without putting tokens in tool arguments:

```toml
[gateways.solveit]
url = "https://solveit.example.com/gate"
token_env = "SOLVEIT_TOKEN"
verify = false   # optional: accept a self-signed certificate
```


## The stream protocol

Run `clikernel` as a plain CLI process for clients that read a text stream rather than MCP messages. It uses a delimiter-framed stdin/stdout protocol:

- Input is not echoed.
- Each request gets a `.` acknowledgement.
- A per-process random delimiter marks the end of each response.
- Multiline cells are framed by `--` and the delimiter.

The startup banner supplies the protocol instructions and delimiter. Running `clikernel` without arguments creates a kernel and stops it on exit. `--kernel <id>` attaches to an existing kernel and leaves it running on exit.
